# Bahdanau Attention

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class BahdanauAttention(nn.Module):
  def __init__(self, hidden_size):
   super().__init__()
   self.W_a = nn.Linear(hidden_size, hidden_size) #Decoder (s)
   self.U_a = nn.Linear(hidden_size, hidden_size) #Encoder (h)
   self.v_a = nn.Linear(hidden_size, 1) # Final hs

  def forward(self, decoder_hidden, encoder_outputs):
    decoder_hidden = decoder_hidden.unsqueeze(1)
    #Step-1 : Score Calculation
    scores = self.v_a(torch.tanh(
        self.W_a(decoder_hidden) + self.U_a(encoder_outputs)
    ))
    scores = scores.squeeze(-1)
    #Step-2: Weights

    atten_weights = F.softmax(scores, dim=1)

    #Step-3: Context Vector
    context = torch.bmm(atten_weights.unsqueeze(1), encoder_outputs)
    context = context.squeeze(1) # Fix: squeeze and assign back

    return context, atten_weights

In [2]:
class EncoderStep(nn.Module):
  def __init__(self ,vocab_size, embedding_dim,hidden_size,num_layers):
    super().__init__()
    self.embeddings = nn.Embedding(vocab_size, embedding_dim)
    self.gru = nn.GRU(embedding_dim, hidden_size, num_layers, batch_first=True)

  def forward(self, input):
    embedded = self.embeddings(input)
    output, hidden = self.gru(embedded)
    return output

In [3]:
encode = EncoderStep(120, 50, 64, 2)
dummy_input = torch.randint(0, 120, (10, 5))   # (batch=10, seq_len=5)
outputs = encode(dummy_input)
print(outputs.shape)

torch.Size([10, 5, 64])


In [4]:
decoder_dummy = torch.randn(10, 64)
bahdanau = BahdanauAttention(64)

context, attn_weights = bahdanau(decoder_dummy, outputs)

print(context.shape)        # expected: (10, 64)
print(attn_weights.shape)   # expected: (10, 5)

torch.Size([10, 64])
torch.Size([10, 5])


In [5]:
print(attn_weights.sum(dim=1))

tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000], grad_fn=<SumBackward1>)


# Luoung Attention

In [11]:
class LuoungAttention(nn.Module):
  def __init__(self, hidden_size):
    super().__init__()

    self.W_a = nn.Linear(hidden_size, hidden_size)

  def forward(self, encoder_outputs, decoder_hidden):
    # Step-1
    transformed = self.W_a(encoder_outputs)
    # Step-2
    scores = torch.bmm(transformed, decoder_hidden.unsqueeze(2))
    scores = scores.squeeze(2)
    # Step-3
    attn_weights = F.softmax(scores, dim=1)
    # Step-4
    context = torch.bmm(attn_weights.unsqueeze(1), encoder_outputs)
    context = context.squeeze(1)
    return context, attn_weights

In [13]:
batch, seq_len, hidden = 10, 5, 64

encoder_outputs = torch.randn(batch, seq_len, hidden)
decoder_hidden  = torch.randn(batch, hidden)      # s_i (current)

attn = LuoungAttention(hidden)
context, weights = attn(encoder_outputs, decoder_hidden)

print(context.shape)
print(weights.shape)
print(weights.sum(1))

torch.Size([10, 64])
torch.Size([10, 5])
tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000], grad_fn=<SumBackward1>)
